# Lesson 2.4 — Integrating MCP Servers into Claude Code & Agent Workflows

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Choose project-level (.mcp.json) vs user-level (~/.claude.json) MCP server scope
- Use environment variable expansion for credentials without committing secrets
- Understand that all configured servers' tools are available simultaneously
- Expose content catalogs as MCP resources to reduce exploratory tool calls

**You will produce:** A project .mcp.json with ${ENV} expansion plus notes on resources and user-scope servers.

## Concepts

**MCP server scoping:**
- **Project-level `.mcp.json`** (committed) for shared team tooling.
- **User-level `~/.claude.json`** for personal/experimental servers (not shared via VCS).

**Credentials:** use **environment variable expansion** in `.mcp.json` (e.g. `${GITHUB_TOKEN}`) so secrets are never committed.

**Discovery:** tools from **all** configured MCP servers are discovered at connection time and available **simultaneously** to the agent.

**Resources:** MCP **resources** expose content catalogs (issue summaries, docs hierarchies, DB schemas) so the agent has visibility into available data **without exploratory tool calls**.

Also: enhance MCP tool descriptions so the agent prefers them over built-ins like Grep; and prefer existing community servers (e.g. Jira) over custom ones for standard integrations.

> **Reference:** [Anthropic - MCP in Claude Code](https://docs.anthropic.com/en/docs/claude-code/mcp)

In [ ]:
import os, json, pathlib
WORK = pathlib.Path('mcp_demo'); WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print('working dir:', os.getcwd())

## Code-Along: Step 1 — Write a project-scoped .mcp.json with env expansion

`${GITHUB_TOKEN}` is expanded at runtime; the token never lands in version control. This file is committed and shared with the team.

In [ ]:
mcp_config = {
    "mcpServers": {
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "${GITHUB_TOKEN}"}
        },
        "company-db": {
            "command": "python",
            "args": ["-m", "company_mcp.server"],
            "env": {"DB_URL": "${DB_URL}"}
        }
    }
}
with open('.mcp.json', 'w', encoding='utf-8') as f:
    json.dump(mcp_config, f, indent=2)
print(open('.mcp.json').read())

## Code-Along: Step 2 — Verify env expansion safely

We simulate expansion so you can confirm no literal secret is stored in the file.

In [ ]:
import re
raw = open('.mcp.json').read()
placeholders = re.findall(r'\$\{(\w+)\}', raw)
print('env placeholders (expanded at runtime):', placeholders)
assert 'ghp_' not in raw, 'No literal token should be committed!'
print('OK: no literal secrets committed')

## Code-Along: Step 3 — Model an MCP resource catalog

Resources let the agent see what data exists without probing. Here is a small catalog the server would expose.

In [ ]:
resources = [
    {"uri": "issues://open", "name": "Open issues summary"},
    {"uri": "schema://orders", "name": "Orders table schema"},
    {"uri": "docs://api", "name": "API documentation hierarchy"},
]
print(json.dumps(resources, indent=2))
# Exposing these avoids exploratory tool calls to discover available data.

## Exercise

Add a personal/experimental server to a modeled `~/.claude.json` (user scope) and show both project and user servers would be available simultaneously. Then write an assertion that your `.mcp.json` contains zero literal credentials (only `${...}` placeholders).

In [ ]:
# TODO: build a user_config dict with an experimental server and dump to 'user_claude.json'
# TODO: merge server names from both configs and print the combined tool-source list
# TODO: assert re.search(r'ghp_|sk-', open('.mcp.json').read()) is None

## Key Takeaways

- Project .mcp.json for shared team servers; user ~/.claude.json for personal/experimental ones.
- Use ${ENV} expansion so credentials never get committed.
- All configured MCP servers' tools are discovered and available simultaneously.
- Expose content catalogs as MCP resources to cut exploratory tool calls.

---

**Next:** [Lesson 2.5 — Selecting Built-in Tools Effectively](./lesson-2.5-built-in-tools.ipynb)